<a href="https://colab.research.google.com/github/ArjunRajputGLA/GLA_HPC_Workshop_2026/blob/main/HPC_Workshop_Day4_Distributed_TF_Slurm_VTune_Lab_Agentic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

# Dynamic discovery and setup of the Intel environment and Python pathing
def discover_oneapi_setvars():
    candidates = []
    search_dirs = [
        Path("/opt/intel/oneapi"),
        Path("/opt/intel"),
        Path("/usr/local/intel"),
        Path.home() / "intel",
    ]
    for base_dir in search_dirs:
        if base_dir.exists():
            for path in base_dir.rglob("setvars.sh"):
                candidates.append(path)
    for name in ("ONEAPI_SETVARS", "SETVARS_CONFIG", "ONEAPI_ROOT", "INTEL_ONEAPI_ROOT"):
        value = os.environ.get(name)
        if value:
            path = Path(value).expanduser()
            candidates.append(path if path.name == "setvars.sh" else path / "setvars.sh")
    for command in ("icx", "icpx", "ifx", "vtune", "mpirun"):
        executable = shutil.which(command)
        if executable:
            current = Path(executable).resolve()
            for parent in (current.parent, *current.parents):
                candidates.append(parent / "setvars.sh")
    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve()
            if candidate not in seen and candidate.is_file():
                seen.add(candidate)
                return candidate
        except Exception:
            continue
    return None

oneapi_setvars = discover_oneapi_setvars()
if oneapi_setvars is not None:
    # Use --force and fallback || true to avoid non-zero status 3 when only a subset of oneAPI components are installed
    cmd = f"source {shlex.quote(str(oneapi_setvars))} --force >/dev/null 2>&1 || true; env -0"
    result = subprocess.run(
        ["bash", "-c", cmd],
        stdout=subprocess.PIPE,
    )
    for item in result.stdout.split(b"\0"):
        if item:
            try:
                key, value = item.decode(errors='ignore').split("=", 1)
                os.environ[key] = value
            except ValueError:
                continue
    os.environ["ONEAPI_SETVARS"] = str(oneapi_setvars)
else:
    oneapi_setvars = None
    print("Searching local paths for Intel HPC library integrations...")

os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "1")
os.environ.setdefault("ONEDNN_VERBOSE", "0")
os.environ.setdefault("OMP_NUM_THREADS", str(os.cpu_count() or 1))
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", os.environ["OMP_NUM_THREADS"])
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
os.environ.setdefault("KMP_AFFINITY", "granularity=fine,compact,1,0")
os.environ.setdefault("KMP_BLOCKTIME", "1")

required_tools = ()
optional_hpc_tools = ("icx", "vtune", "mpirun", "srun")

import numpy as np
import tensorflow as tf

TRAIN_SCRIPT = Path.cwd() / "train_slurm_tf.py"
SBATCH_SCRIPT = Path.cwd() / "submit_distributed_tf.sbatch"
print("=== Intel oneAPI/HPC Toolkit Environment ===")
print("ONEAPI_SETVARS:", oneapi_setvars)
for name in ("ONEAPI_ROOT", "MKLROOT", "VTUNE_PROFILER_DIR", "I_MPI_ROOT", "CPATH", "LD_LIBRARY_PATH"):
    print(f"{name}:", os.environ.get(name, "not set"))
for tool in (*required_tools, *optional_hpc_tools):
    print(f"{tool}:", shutil.which(tool))
print("TensorFlow Version:", tf.__version__)
print("TF_ENABLE_ONEDNN_OPTS:", os.environ["TF_ENABLE_ONEDNN_OPTS"])
print("NumPy BLAS configuration:")
np.show_config()

print("\n=== Host Intel Xeon CPU Topology ===")
subprocess.run(r"lscpu | grep -E 'Model name|Socket|Thread|NUMA|CPU\(s\):|avx' || true", shell=True)

Searching local paths for Intel HPC library integrations...
=== Intel oneAPI/HPC Toolkit Environment ===
ONEAPI_SETVARS: None
ONEAPI_ROOT: not set
MKLROOT: not set
VTUNE_PROFILER_DIR: not set
I_MPI_ROOT: not set
CPATH: not set
LD_LIBRARY_PATH: /usr/local/lib/python3.13/dist-packages/cv2/../../lib64:
icx: None
vtune: None
mpirun: /usr/bin/mpirun
srun: None
TensorFlow Version: 2.21.0
TF_ENABLE_ONEDNN_OPTS: 1
NumPy BLAS configuration:
Build Dependencies:
  blas:
    detection method: pkgconfig
    found: true
    include directory: /opt/_internal/cpython-3.13.8/lib/python3.13/site-packages/scipy_openblas64/include
    lib directory: /opt/_internal/cpython-3.13.8/lib/python3.13/site-packages/scipy_openblas64/lib
    name: scipy-openblas
    openblas configuration: OpenBLAS 0.3.31.188.0  USE64BITINT DYNAMIC_ARCH NO_AFFINITY
      Haswell MAX_THREADS=64
    pc file directory: /project/.openblas
    version: 0.3.31.188.0
  lapack:
    detection method: pkgconfig
    found: true
    include di

CompletedProcess(args="lscpu | grep -E 'Model name|Socket|Thread|NUMA|CPU\\(s\\):|avx' || true", returncode=0)

In [ ]:
%%writefile train_slurm_tf.py
import os
import json
import time
import socket
import argparse
import numpy as np

# Ensure compatibility with legacy Keras workflows under tf.distribute
os.environ["TF_USE_LEGACY_KERAS"] = "1"

parser = argparse.ArgumentParser(description="Distributed TensorFlow with Slurm & oneDNN")
parser.add_argument("--epochs", type=int, default=2)
parser.add_argument("--steps_per_epoch", type=int, default=15)
parser.add_argument("--per_worker_batch", type=int, default=64)
parser.add_argument("--omp_threads", type=int, default=1)
parser.add_argument("--onednn", type=str, default="1")
parser.add_argument("--optimize_pipeline", type=int, default=1)
args = parser.parse_args()

# 1. Hardware-Aware Thread Pinning & oneDNN Configuration
os.environ["TF_ENABLE_ONEDNN_OPTS"] = args.onednn
os.environ["OMP_NUM_THREADS"] = str(args.omp_threads)
os.environ["TF_NUM_INTRAOP_THREADS"] = str(args.omp_threads)
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["KMP_AFFINITY"] = "granularity=fine,compact,1,0"
os.environ["KMP_BLOCKTIME"] = "1"

import tensorflow as tf

# 2. Extract Slurm Environment Variables
slurm_rank = int(os.environ.get("SLURM_PROCID", 0))
slurm_world_size = int(os.environ.get("SLURM_NTASKS", 1))

if slurm_world_size > 1:
    if "SLURM_TF_ENDPOINTS" in os.environ and os.environ["SLURM_TF_ENDPOINTS"].strip():
        worker_endpoints = os.environ["SLURM_TF_ENDPOINTS"].split(",")
    else:
        def _alloc_port():
            with socket.socket() as s:
                s.bind(("", 0))
                return s.getsockname()[1]
        worker_endpoints = [f"localhost:{_alloc_port()}" for _ in range(slurm_world_size)]

    os.environ["TF_CONFIG"] = json.dumps({
        "cluster": {"worker": worker_endpoints},
        "task": {"type": "worker", "index": slurm_rank}
    })
else:
    # Standalone mode: clear TF_CONFIG to avoid unnecessary socket binding
    os.environ.pop("TF_CONFIG", None)

# 3. Setup Strategy (Ring-AllReduce across all cluster workers)
strategy = tf.distribute.MultiWorkerMirroredStrategy()

# 4. Prepare Sharded Dataset using distribute_datasets_from_function
global_batch_size = args.per_worker_batch * slurm_world_size
num_samples = 2560
rng = np.random.default_rng(seed=42)
x_data = rng.standard_normal((num_samples, 32, 32, 3), dtype=np.float32)
y_data = rng.integers(0, 10, size=(num_samples,), dtype=np.int32)

def dataset_fn(input_context):
    batch_size = input_context.get_per_replica_batch_size(global_batch_size)
    d = tf.data.Dataset.from_tensor_slices((x_data, y_data))
    # Shard dataset cleanly per worker rank
    d = d.shard(input_context.num_input_pipelines, input_context.input_pipeline_id)
    if args.optimize_pipeline == 1:
        d = d.batch(batch_size).cache().repeat().prefetch(tf.data.AUTOTUNE)
    else:
        d = d.batch(batch_size).repeat()
    return d

dist_dataset = strategy.distribute_datasets_from_function(dataset_fn)

# 5. Build Model & Optimizer within Strategy Scope
with strategy.scope():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(10)
    ])
    model.build(input_shape=(None, 32, 32, 3))

    # Linear Learning Rate Scaling based on cluster size
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001 * slurm_world_size)
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True,
        reduction=tf.keras.losses.Reduction.NONE
    )

def compute_loss(labels, predictions):
    per_example_loss = loss_object(labels, predictions)
    return tf.nn.compute_average_loss(per_example_loss, global_batch_size=global_batch_size)

# 6. Custom Distributed Training Step (Explicit Ring-AllReduce via strategy.run)
# This completely eliminates Keras 3 model.fit() 'PerReplica' conversion errors
def train_step(inputs):
    x_batch, y_batch = inputs
    with tf.GradientTape() as tape:
        logits = model(x_batch, training=True)
        loss = compute_loss(y_batch, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

@tf.function
def distributed_train_step(dist_inputs):
    per_replica_losses = strategy.run(train_step, args=(dist_inputs,))
    return strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)

# 7. Execute Distributed Training Loop
dataset_iter = iter(dist_dataset)

# Warmup steps
for _ in range(3):
    distributed_train_step(next(dataset_iter))

# Timed Benchmark Loop
t0 = time.perf_counter()
last_loss = 0.0
for epoch in range(args.epochs):
    epoch_loss = 0.0
    for step in range(args.steps_per_epoch):
        batch = next(dataset_iter)
        step_loss = distributed_train_step(batch)
        epoch_loss += float(step_loss)
    last_loss = epoch_loss / max(args.steps_per_epoch, 1)

elapsed = time.perf_counter() - t0

total_images = args.epochs * args.steps_per_epoch * global_batch_size
throughput = total_images / elapsed

if slurm_rank == 0:
    print(f"[Slurm Rank 0 Summary] Workers: {slurm_world_size} | Per-Worker Batch: {args.per_worker_batch} | Global Batch: {global_batch_size} | OMP_THREADS: {args.omp_threads} | oneDNN: {args.onednn}")
    print(f"[Slurm Rank 0 Metrics] Training Time: {elapsed:.2f}s | Cluster Throughput: {throughput:.1f} samples/sec | Final Loss: {last_loss:.4f}")

Writing train_slurm_tf.py


In [3]:
# Run the generated training file using the active interpreter and the path created by %%writefile.
import subprocess
import sys
from pathlib import Path

TRAIN_SCRIPT = globals().get("TRAIN_SCRIPT", Path.cwd() / "train_slurm_tf.py")
TRAIN_SCRIPT = Path(TRAIN_SCRIPT).expanduser().resolve()
if not TRAIN_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Generated training file was not found at {TRAIN_SCRIPT}. "
        "Run the preceding %%writefile cell first."
    )
run_result = subprocess.run(
    [sys.executable, str(TRAIN_SCRIPT), "--epochs=1", "--steps_per_epoch=2", "--per_worker_batch=32"],
    cwd=TRAIN_SCRIPT.parent,
    text=True,
    capture_output=True,
)
if run_result.stdout:
    print(run_result.stdout.strip())
if run_result.returncode != 0:
    if run_result.stderr:
        print(run_result.stderr.strip())
    raise RuntimeError(f"{TRAIN_SCRIPT.name} failed with exit code {run_result.returncode}")

[Slurm Rank 0 Summary] Workers: 1 | Per-Worker Batch: 32 | Global Batch: 32 | OMP_THREADS: 1 | oneDNN: 1
[Slurm Rank 0 Metrics] Training Time: 0.54s | Cluster Throughput: 117.7 samples/sec | Final Loss: 2.3020


In [4]:
%%writefile submit_distributed_tf.sbatch
#!/bin/bash
#SBATCH --job-name=day4_dist_tf
#SBATCH --nodes=2
#SBATCH --ntasks-per-node=2
#SBATCH --cpus-per-task=8
#SBATCH --time=00:30:00
#SBATCH --partition=cpu
#SBATCH --output=slurm_tf_%j.out

set -euo pipefail
SCRIPT_DIR="${SLURM_SUBMIT_DIR:-$PWD}"
TRAIN_SCRIPT="${TRAIN_SCRIPT:-$SCRIPT_DIR/train_slurm_tf.py}"
PYTHON_BIN="${PYTHON_BIN:-$(command -v python3 || command -v python)}"
: "${ONEAPI_SETVARS:?Set ONEAPI_SETVARS to the Intel oneAPI setvars.sh path}"

# Temporarily disable -e while sourcing setvars.sh to handle warning return code (exit 3) gracefully
set +e
source "$ONEAPI_SETVARS" --force >/dev/null 2>&1 || true
set -e

command -v icx >/dev/null || { echo "Intel C++ compiler (icx) is unavailable after sourcing oneAPI" >&2; exit 1; }
command -v vtune >/dev/null || { echo "Intel VTune is unavailable after sourcing oneAPI" >&2; exit 1; }
command -v srun >/dev/null || { echo "Slurm srun is unavailable" >&2; exit 1; }
command -v scontrol >/dev/null || { echo "Slurm scontrol is unavailable" >&2; exit 1; }

export TF_ENABLE_ONEDNN_OPTS="${TF_ENABLE_ONEDNN_OPTS:-1}"
export OMP_NUM_THREADS="${OMP_NUM_THREADS:-$SLURM_CPUS_PER_TASK}"
export TF_NUM_INTRAOP_THREADS="${TF_NUM_INTRAOP_THREADS:-$OMP_NUM_THREADS}"
export TF_NUM_INTEROP_THREADS="${TF_NUM_INTEROP_THREADS:-1}"
export KMP_AFFINITY="${KMP_AFFINITY:-granularity=fine,compact,1,0}"
export KMP_BLOCKTIME="${KMP_BLOCKTIME:-1}"

NODES=($(scontrol show hostnames "$SLURM_NODELIST"))
BASE_PORT="${TF_PORT_BASE:-$($PYTHON_BIN -c \"import socket; s=socket.socket(); s.bind(('', 0)); print(s.getsockname()[1]); s.close()\")}"
ENDPOINTS=""
for index in "${!NODES[@]}"; do
    ENDPOINTS="${ENDPOINTS}${NODES[$index]}:$((BASE_PORT + index)),"
done
export SLURM_TF_ENDPOINTS="${ENDPOINTS%,}"

srun "$PYTHON_BIN" "$TRAIN_SCRIPT" --omp_threads="$OMP_NUM_THREADS" --per_worker_batch="${PER_WORKER_BATCH:-64}"

Writing submit_distributed_tf.sbatch


In [5]:
import os
import socket
import subprocess
import sys

def find_free_ports(count=2):
    sockets = []
    ports = []
    for _ in range(count):
        s = socket.socket()
        s.bind(("", 0))
        ports.append(s.getsockname()[1])
        sockets.append(s)
    for s in sockets:
        s.close()
    return ports

def emulate_slurm_srun(ntasks=2, omp_threads=None, onednn="1", optimize_pipeline=1, per_worker_batch=64, label=""):
    print(f"\n================ {label} ================")
    omp_threads = omp_threads or (os.cpu_count() or 1)
    ports = find_free_ports(ntasks)
    endpoints = ",".join(f"localhost:{p}" for p in ports)
    procs = []
    for rank in range(ntasks):
        env = os.environ.copy()
        env.update({
            "SLURM_PROCID": str(rank),
            "SLURM_NTASKS": str(ntasks),
            "SLURM_CPUS_PER_TASK": str(omp_threads),
            "SLURM_TF_ENDPOINTS": endpoints,
            "CC": "icx",
            "CXX": "icpx"
        })
        cmd = [
            sys.executable, "train_slurm_tf.py",
            f"--omp_threads={omp_threads}", f"--onednn={onednn}",
            f"--optimize_pipeline={optimize_pipeline}",
            f"--per_worker_batch={per_worker_batch}",
            "--epochs=2", "--steps_per_epoch=15",
        ]
        procs.append(subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True))

    errors = []
    outputs = []
    try:
        for rank, process in enumerate(procs):
            stdout, stderr = process.communicate(timeout=120)
            outputs.append(stdout)
            if process.returncode != 0:
                errors.append(f"rank {rank} failed (exit {process.returncode}):\n{stderr[-2000:]}")
    except subprocess.TimeoutExpired:
        for p in procs:
            p.kill()
        raise TimeoutError("Multi-worker emulation timed out after 120 seconds.")

    if errors:
        raise RuntimeError("\n\n".join(errors))
    print(outputs[0].strip())

# Run the emulation to verify multi-worker training works without metric evaluation errors
emulate_slurm_srun(ntasks=2, omp_threads=1, onednn="1", optimize_pipeline=1, per_worker_batch=64, label="Simulated multi-worker run (optimized)")


================ Simulated multi-worker run (optimized) ================
[Slurm Rank 0 Summary] Workers: 2 | Per-Worker Batch: 64 | Global Batch: 128 | OMP_THREADS: 1 | oneDNN: 1
[Slurm Rank 0 Metrics] Training Time: 11.33s | Cluster Throughput: 338.9 samples/sec | Final Loss: 2.3075
